In [14]:
# cái này để install kaggle hub giúp tải data từ kaggle

%pip install pandas numpy kagglehub scikit-learn torch

  Using cached torch-2.2.2-cp312-none-macosx_10_9_x86_64.whl.metadata (25 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.2.2-cp312-none-macosx_10_9_x86_64.whl (150.8 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart t

In [1]:
# để import ở đây hết nhé mọi người

import glob
import os

import kagglehub
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

c:\Users\dummyking\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---- Option B: set this to your CSV file to skip the download (leave None to use Option A) ----
MANUAL_CSV_PATH = None     # e.g. r"C:\Users\you\Downloads\GOOGL.csv"

DATASET = "varpit94/google-stock-data"  # link của cái chỗ tải trên kaggle

if MANUAL_CSV_PATH is None:   # chỉ chạy khi không có file csv, nếu có thì bỏ qua bước download
    # kagglehub.login()    # uncomment if it asks you to authenticate
    dataset_dir = kagglehub.dataset_download(DATASET)
    print("Downloaded to:", dataset_dir)
else:
    dataset_dir = os.path.dirname(MANUAL_CSV_PATH)
    print("Using manual path folder:", dataset_dir)

Downloaded to: C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6


In [3]:
if MANUAL_CSV_PATH is not None:
    csv_files = [MANUAL_CSV_PATH]
else:
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True))

if not csv_files:
    raise FileNotFoundError(f"No CSV found in {dataset_dir}. Download it manually and set MANUAL_CSV_PATH.")

for i, f in enumerate(csv_files):
    print(i, "->", f, f"({os.path.getsize(f)/1024:.1f} KB)")

0 -> C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6\GOOGL.csv (325.7 KB)


In [4]:
# cái này sẽ đọc csv và trả về dataframe nek 

CSV_PATH = csv_files[0]
print("Reading:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Shape (rows, columns):", df.shape)
df.head()

Reading: C:\Users\dummyking\.cache\kagglehub\datasets\varpit94\google-stock-data\versions\6\GOOGL.csv
Shape (rows, columns): (4431, 7)


,Date,Open,High,Low,Close,Adj Close,Volume
0,2004-08-19,50.050049,52.082081,48.028027,50.220219,50.220219,44659096
1,2004-08-20,50.555557,54.594597,50.300301,54.209209,54.209209,22834343
2,2004-08-23,55.430431,56.796799,54.579578,54.754753,54.754753,18256126
3,2004-08-24,55.675674,55.855858,51.836838,52.487488,52.487488,15247337
4,2004-08-25,52.532532,54.054054,51.991993,53.053055,53.053055,9188602


In [5]:
# kiểm tra cấu trúc của files

print("Columns:", list(df.columns))
print()
df.info()

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

<class 'pandas.DataFrame'>
RangeIndex: 4431 entries, 0 to 4430
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       4431 non-null   str    
 1   Open       4431 non-null   float64
 2   High       4431 non-null   float64
 3   Low        4431 non-null   float64
 4   Close      4431 non-null   float64
 5   Adj Close  4431 non-null   float64
 6   Volume     4431 non-null   int64  
dtypes: float64(5), int64(1), str(1)
memory usage: 285.7 KB


In [6]:
print("Summary statistics:")
display(df.describe())  # dùng chức năng describe để xem các thông số

Summary statistics:


,Open,High,Low,Close,Adj Close,Volume
count,4431.000000,4431.000000,4431.000000,4431.000000,4431.000000,4.431000e+03
mean,693.087345,699.735595,686.078751,693.097367,693.097367,6.444992e+06
std,645.118799,651.331215,638.579488,645.187806,645.187806,7.690351e+06
min,49.644646,50.920921,48.028027,50.055054,50.055054,4.656000e+05
25%,248.558563,250.853355,245.813309,248.415916,248.415916,1.695600e+06
50%,434.924927,437.887878,432.687683,435.330322,435.330322,3.778418e+06
75%,1007.364990,1020.649994,997.274994,1007.790008,1007.790008,8.002390e+06
max,3025.000000,3030.929932,2977.979980,2996.770020,2996.770020,8.215117e+07


In [7]:
# kiểm tra dữ liệu có bị thiếu hay không, có bị trùng lặp hay không, và nếu có cột Date thì in ra ngày đầu tiên và ngày cuối cùng

print("Missing values per column:")
display(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())
if "Date" in df.columns:
    print("First date in file:", df["Date"].iloc[0])
    print("Last date in file :", df["Date"].iloc[-1])

Missing values per column:


Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Duplicate rows: 0
First date in file: 2004-08-19
Last date in file : 2022-03-24


In [8]:
# Tạo cửa sổ 30 ngày liên tiếp

WINDOW = 30  # dùng 30 ngày liên tiếp để dự đoán ngày kế tiếp

# Sort theo ngày tăng dần
df["Date"] = pd.to_datetime(df["Date"])
df_sorted = df.sort_values("Date").reset_index(drop=True)

# Chọn feature, lấy ra mảng số
FEATURES = ["Open", "High", "Low", "Close", "Volume"]   # bỏ Adj Close vì giống hệt Close
data = df_sorted[FEATURES].values.astype(np.float32)    # shape (số_ngày, số_feature)
TARGET_COL = FEATURES.index("Close")                    # cột cần dự đoán


# Hàm tạo cửa sổ
def make_windows(data, target_col, window=WINDOW):
    """
    data       : mảng 2D (số_ngày, số_feature), đã sort theo ngày
    target_col : chỉ số cột cần dự đoán (Close)
    return     : X (số_sample, window, số_feature), y (số_sample, 1)
    """
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])           # 30 ngày liên tiếp: i .. i+29
        y.append(data[i + window, target_col])   # Close của ngày kế tiếp: i+30
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32).reshape(-1, 1)
    return X, y


X, y = make_windows(data, TARGET_COL)

# Kiểm tra input và dimension
print("data shape:", data.shape)
print("X shape:", X.shape, "-> (số sample, 30 ngày, số feature)")
print("y shape:", y.shape, "-> (số sample, 1)")

assert X.shape == (len(data) - WINDOW, WINDOW, data.shape[1])
assert y.shape == (len(data) - WINDOW, 1)
assert np.isfinite(X).all() and np.isfinite(y).all()      # không có NaN / inf
assert np.array_equal(X[0], data[0:30])                   # sample đầu: ngày 0..29
assert y[0, 0] == data[30, TARGET_COL]                    # nhãn = Close ngày 30
assert np.array_equal(X[1][:-1], X[0][1:])                # cửa sổ trượt đúng 1 ngày
assert y[-1, 0] == data[-1, TARGET_COL]                   # sample cuối dùng ngày cuối
print("Tất cả kiểm tra đều OK")

data shape: (4431, 5)
X shape: (4401, 30, 5) -> (số sample, 30 ngày, số feature)
y shape: (4401, 1) -> (số sample, 1)
Tất cả kiểm tra đều OK


chia data

In [9]:
TRAIN_RATIO = 0.70            # 70% đầu làm train
VAL_RATIO   = 0.15            # 15% kế làm val (test = phần còn lại = 15%)

n = len(X)                    # tổng số sample (4401)
n_train = int(n * TRAIN_RATIO)    # số sample train, int() để bỏ phần thập phân
n_val   = int(n * VAL_RATIO)      # số sample val

# Cắt theo thứ tự thời gian: [đầu : n_train], [n_train : n_train+n_val], [n_train+n_val : cuối]
X_train, y_train = X[:n_train],                 y[:n_train]
X_val,   y_val   = X[n_train:n_train + n_val],  y[n_train:n_train + n_val]
X_test,  y_test  = X[n_train + n_val:],         y[n_train + n_val:]

print("train:", X_train.shape, y_train.shape)
print("val  :", X_val.shape,   y_val.shape)
print("test :", X_test.shape,  y_test.shape)
assert len(X_train) + len(X_val) + len(X_test) == n   # cộng lại phải đủ n, không mất sample

train: (3080, 30, 5) (3080, 1)
val  : (660, 30, 5) (660, 1)
test : (661, 30, 5) (661, 1)


chuẩn hóa min max

In [10]:
n_features = X.shape[2]          # số feature = 5 (Open, High, Low, Close, Volume)
scaler = MinMaxScaler()          # tạo scaler rỗng
scaler.fit(X_train.reshape(-1, n_features))
# reshape (3080,30,5) -> (92400,5): scaler chỉ nhận mảng 2D, mỗi cột là 1 feature
# fit = học min và max của từng cột, CHỈ từ train

def scale_X(arr):
    # transform: áp công thức (x-min)/(max-min) bằng min/max đã học từ train
    # reshape(-1,5) để đưa về 2D, rồi reshape(arr.shape) để trả về dạng 3D ban đầu
    return scaler.transform(arr.reshape(-1, n_features)).reshape(arr.shape).astype(np.float32)

close_min = scaler.data_min_[TARGET_COL]   # min của riêng cột Close (từ train)
close_max = scaler.data_max_[TARGET_COL]   # max của riêng cột Close (từ train)

def scale_y(arr):
    # y là giá Close nên dùng min/max của Close, cùng công thức (x-min)/(max-min)
    return ((arr - close_min) / (close_max - close_min)).astype(np.float32)

def inverse_y(arr):
    # đổi ngược về giá thật ($): x = scaled*(max-min) + min
    return arr * (close_max - close_min) + close_min

X_train_s, X_val_s, X_test_s = scale_X(X_train), scale_X(X_val), scale_X(X_test)
y_train_s, y_val_s, y_test_s = scale_y(y_train),  scale_y(y_val),  scale_y(y_test)
# đuôi _s = scaled. Cả 3 tập dùng chung min/max của train

print("train sau scale: min", X_train_s.min(), "max", X_train_s.max())
print("val   sau scale: max", X_val_s.max())
print("test  sau scale: max", X_test_s.max())
assert np.allclose(inverse_y(y_train_s), y_train, atol=1e-3)
# scale rồi đổi ngược phải ra lại giá cũ (sai số nhỏ hơn 0.001)

train sau scale: min 0.0 max 1.0
val   sau scale: max 1.5860618
test  sau scale: max 3.7813575


In [11]:
class StockDataset(Dataset):                 # kế thừa Dataset của PyTorch
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)         # numpy -> tensor, shape (số_sample, 30, 5)
        self.y = torch.from_numpy(y)         # numpy -> tensor, shape (số_sample, 1)

    def __len__(self):
        return len(self.X)                   # DataLoader cần biết tổng số sample

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]      # trả 1 cặp (cửa sổ 30 ngày, Close ngày kế)

train_ds = StockDataset(X_train_s, y_train_s)
val_ds   = StockDataset(X_val_s,   y_val_s)
test_ds  = StockDataset(X_test_s,  y_test_s)

BATCH_SIZE = 32                              # mỗi lần đưa 32 sample vào model
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)   # train: xáo
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)  # val: giữ thứ tự
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)  # test: giữ thứ tự

xb, yb = next(iter(train_loader))            # lấy thử 1 batch đầu tiên
print("1 batch X:", xb.shape, "| y:", yb.shape)   # mong đợi (32,30,5) và (32,1)
assert xb.shape == (BATCH_SIZE, WINDOW, n_features) and yb.shape == (BATCH_SIZE, 1)
print("Số sample: train", len(train_ds), "| val", len(val_ds), "| test", len(test_ds))

1 batch X: torch.Size([32, 30, 5]) | y: torch.Size([32, 1])
Số sample: train 3080 | val 660 | test 661


In [12]:
class StockRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # Hai tầng RNN giúp học đặc trưng chuỗi sâu hơn; batch_first nhận dạng (batch, ngày, feature).
        self.rnn = nn.RNN(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # Attention học trọng số quan trọng cho mỗi ngày trong cửa sổ.
        self.attention = nn.Linear(hidden_size, 1)
        # Dropout giảm overfitting; Linear đổi vector ẩn thành một giá Close dự đoán.
        self.output_layer = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        sequence, _ = self.rnn(x)  # Vector biểu diễn của từng ngày; trạng thái ẩn cuối không dùng riêng.
        weights = torch.softmax(self.attention(sequence), dim=1)  # Chuẩn hóa trọng số attention theo các ngày.
        context = (sequence * weights).sum(dim=1)  # Tổng có trọng số, gộp 30 ngày thành một vector.
        return self.output_layer(context)  # Dự đoán Close đã chuẩn hóa, shape (batch, 1).


model = StockRNN(input_size=n_features)
prediction = model(xb)
print("Output shape:", prediction.shape)  # mong đợi (batch_size, 1)
assert prediction.shape == yb.shape

Output shape: torch.Size([32, 1])


In [13]:
# Train the model and monitor validation loss.
# Chọn GPU phù hợp nếu có, nếu không thì chạy trên CPU.

model = model.to(device)

criterion = nn.MSELoss()  # Đo sai số bình phương trên giá đã chuẩn hóa.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Cập nhật trọng số của model.

EPOCHS = 50  # Số vòng học tối đa.
PATIENCE = 8  # Dừng nếu validation loss không cải thiện sau 8 epoch.
best_val_loss = float("inf")
epochs_without_improvement = 0
best_state = None
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()  # Bật chế độ train (dropout hoạt động).
    train_loss_sum = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()  # Xóa gradient từ batch trước.
        loss = criterion(model(batch_X), batch_y)
        loss.backward()  # Tính gradient.
        optimizer.step()  # Cập nhật trọng số.
        train_loss_sum += loss.item() * batch_X.size(0)

    epoch_train_loss = train_loss_sum / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()  # Tắt dropout; đánh giá validation mà không cập nhật trọng số.
    val_loss_sum = 0.0
    with torch.no_grad():  # Không tính gradient trong bước validation.
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            val_loss_sum += criterion(model(batch_X), batch_y).item() * batch_X.size(0)

    epoch_val_loss = val_loss_sum / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | train loss: {epoch_train_loss:.6f} | validation loss: {epoch_val_loss:.6f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        # Lưu bản sao trọng số tốt nhất để các epoch sau không ghi đè.
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping: validation loss did not improve.")
            break

model.load_state_dict(best_state)  # Khôi phục trọng số có validation loss thấp nhất.
print(f"Best validation loss: {best_val_loss:.6f}")

Epoch 01/50 | train loss: 0.010639 | validation loss: 0.027016
Epoch 02/50 | train loss: 0.001861 | validation loss: 0.024041
Epoch 03/50 | train loss: 0.001741 | validation loss: 0.016890
Epoch 04/50 | train loss: 0.001703 | validation loss: 0.022525
Epoch 05/50 | train loss: 0.001578 | validation loss: 0.016185
Epoch 06/50 | train loss: 0.001482 | validation loss: 0.014793
Epoch 07/50 | train loss: 0.001519 | validation loss: 0.003954
Epoch 08/50 | train loss: 0.001496 | validation loss: 0.008338
Epoch 09/50 | train loss: 0.001370 | validation loss: 0.003406
Epoch 10/50 | train loss: 0.001462 | validation loss: 0.007001
Epoch 11/50 | train loss: 0.001392 | validation loss: 0.007980
Epoch 12/50 | train loss: 0.001287 | validation loss: 0.003213
Epoch 13/50 | train loss: 0.001408 | validation loss: 0.002870
Epoch 14/50 | train loss: 0.001312 | validation loss: 0.008666
Epoch 15/50 | train loss: 0.001285 | validation loss: 0.005798
Epoch 16/50 | train loss: 0.001204 | validation loss: 0